<a href="https://colab.research.google.com/github/ameliaglover13-cyber/Ocala-Teal-Population-Study-2026/blob/main/Avian_Research_Teal_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import os
import glob

print("Scanning your local Colab browser session for your uploaded file...")

# Look directly in the browser's temporary storage
found_files = glob.glob("/content/ebd_US-FL-083_gnwtea*.txt")
input_ebd_file = None

for file in found_files:
    if not file.endswith('_sampling.txt'):
        input_ebd_file = file
        break

if input_ebd_file is None:
    print("\n⚠️ WAITING: The file is still uploading.")
    print("Please look at the bottom-left circle in your Colab side-panel.")
    print("Once that circle finishes filling up, hit the Play button again!")
else:
    output_clean_file = "/content/ocala_teal_data.csv"
    print(f"\n🎯 File successfully located at: {input_ebd_file}")
    print("Extracting Green-winged Teal data for Ocala parks... Please wait a moment.")

    target_parks = ["Ocala Wetland Recharge Park", "Tuscawilla Park"]
    target_species = "Anas crecca"
    cols_to_keep = ['GLOBAL UNIQUE IDENTIFIER', 'LOCALITY', 'SCIENTIFIC NAME', 'OBSERVATION COUNT', 'OBSERVATION DATE']
    data_types = {k: 'str' for k in cols_to_keep}

    chunk_size = 10000
    filtered_chunks = []

    for chunk in pd.read_csv(input_ebd_file, sep='\t', chunksize=chunk_size, low_memory=False, usecols=cols_to_keep, dtype=data_types):
        matched_rows = chunk[
            (chunk['LOCALITY'].isin(target_parks)) &
            (chunk['SCIENTIFIC NAME'] == target_species)
        ]
        filtered_chunks.append(matched_rows)

    final_data = pd.concat(filtered_chunks)
    final_data.to_csv(output_clean_file, index=False)

    print("\n--------------------------------------------------")
    print(f"🏆 SUCCESS! Extracted {len(final_data)} total Green-winged Teal rows.")
    print("Your clean file is ready!")
    print("--------------------------------------------------")

    # Automatically download the clean spreadsheet straight to your phone gallery/downloads!
    from google.colab import files
    files.download(output_clean_file)

Scanning your local Colab browser session for your uploaded file...

🎯 File successfully located at: /content/ebd_US-FL-083_gnwtea_202410_202604_unv_smp_relMay-2026_unvetted.txt
Extracting Green-winged Teal data for Ocala parks... Please wait a moment.

--------------------------------------------------
🏆 SUCCESS! Extracted 0 total Green-winged Teal rows.
Your clean file is ready!
--------------------------------------------------


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

file_path = "/content/ebd_US-FL-083_gnwtea_202410_202604_unv_smp_relMay-2026_unvetted.txt"

# 1. Test read to check delimiter and columns
try:
    df_preview = pd.read_csv(file_path, sep='\t', nrows=5)
    print("✅ File read successfully!")
    print("Columns available in your file:", df_preview.columns.tolist()[:10]) # Shows first 10 columns
except Exception as e:
    print("❌ Error reading file. Check if the file is fully uploaded:", e)

# 2. Check what locations actually exist in your dataset
try:
    # Read only the location column to save Colab memory
    df_locs = pd.read_csv(file_path, sep='\t', usecols=['LOCALITY ID', 'PROTOCOL TYPE', 'OBSERVATION DATE'])

    # Clean up any accidental spaces
    df_locs['LOCALITY ID'] = df_locs['LOCALITY ID'].astype(str).str.strip()

    # Filter for your two target parks
    target_parks = ['L9783145', 'L35493207']
    filtered_data = df_locs[df_locs['LOCALITY ID'].isin(target_parks)]

    print(f"\n🔍 Found {len(filtered_data)} total checklist rows for your target Ocala parks.")
    print("\nBreakdown by Park ID:")
    print(filtered_data['LOCALITY ID'].value_counts())

    print("\nBreakdown by Protocol Type (Looking for Traveling/Stationary):")
    print(filtered_data['PROTOCOL TYPE'].value_counts())

except Exception as e:
    print("❌ Error analyzing columns:", e)

✅ File read successfully!
Columns available in your file: ['GLOBAL UNIQUE IDENTIFIER', 'LAST EDITED DATE', 'TAXONOMIC ORDER', 'CATEGORY', 'TAXON CONCEPT ID', 'COMMON NAME', 'SCIENTIFIC NAME', 'SUBSPECIES COMMON NAME', 'SUBSPECIES SCIENTIFIC NAME', 'EXOTIC CODE']
❌ Error analyzing columns: Usecols do not match columns, columns expected but not found: ['PROTOCOL TYPE']


In [ ]:
import pandas as pd

file_path = "/content/ebd_US-FL-083_gnwtea_202410_202604_unv_smp_relMay-2026_unvetted.txt"

# This will print every single column header inside your file
df_all_cols = pd.read_csv(file_path, sep='\t', nrows=1)
print("Here are ALL the columns in your file:")
print(df_all_cols.columns.tolist())

Here are ALL the columns in your file:
['GLOBAL UNIQUE IDENTIFIER', 'LAST EDITED DATE', 'TAXONOMIC ORDER', 'CATEGORY', 'TAXON CONCEPT ID', 'COMMON NAME', 'SCIENTIFIC NAME', 'SUBSPECIES COMMON NAME', 'SUBSPECIES SCIENTIFIC NAME', 'EXOTIC CODE', 'OBSERVATION COUNT', 'BREEDING CODE', 'BREEDING CATEGORY', 'BEHAVIOR CODE', 'AGE/SEX', 'COUNTRY', 'COUNTRY CODE', 'STATE', 'STATE CODE', 'COUNTY', 'COUNTY CODE', 'IBA CODE', 'BCR CODE', 'USFWS CODE', 'ATLAS BLOCK', 'LOCALITY', 'LOCALITY ID', 'LOCALITY TYPE', 'LATITUDE', 'LONGITUDE', 'OBSERVATION DATE', 'TIME OBSERVATIONS STARTED', 'OBSERVER ID', 'OBSERVER ORCID ID', 'SAMPLING EVENT IDENTIFIER', 'OBSERVATION TYPE', 'PROTOCOL NAME', 'PROTOCOL CODE', 'PROJECT NAMES', 'PROJECT IDENTIFIERS', 'DURATION MINUTES', 'EFFORT DISTANCE KM', 'EFFORT AREA HA', 'NUMBER OBSERVERS', 'ALL SPECIES REPORTED', 'GROUP IDENTIFIER', 'HAS MEDIA', 'APPROVED', 'REVIEWED', 'REASON', 'CHECKLIST COMMENTS', 'SPECIES COMMENTS', 'Unnamed: 52']


In [ ]:
import pandas as pd

file_path = "/content/ebd_US-FL-083_gnwtea_202410_202604_unv_smp_relMay-2026_unvetted.txt"

# 1. Load the data using the exact column names we verified
print("Reading and filtering eBird data... Please wait.")
df = pd.read_csv(file_path, sep='\t', parse_dates=['OBSERVATION DATE'])

# 2. Filter for your two specific Ocala Parks
target_parks = ['L9783145', 'L35493207']
df_filtered = df[df['LOCALITY ID'].astype(str).str.strip().isin(target_parks)].copy()

# 3. Filter for your exact date range (Nov 1, 2024 - Mar 31, 2025)
start_date = '2024-11-01'
end_date = '2025-03-31'
df_filtered = df_filtered[(df_filtered['OBSERVATION DATE'] >= start_date) & (df_filtered['OBSERVATION DATE'] <= end_date)]

# 4. Filter for official protocols only (Stationary 'P21' and Traveling 'P22')
# This matches eBird's official designations for standard effort-based checklists
df_filtered = df_filtered[df_filtered['PROTOCOL CODE'].isin(['P21', 'P22'])]

# 5. Map the Park IDs to their real names for easy reading in your paper
park_names = {
    'L9783145': 'Ocala Wetland Recharge Park',
    'L35493207': 'Tuscawilla Park'
}
df_filtered['Park Name'] = df_filtered['LOCALITY ID'].map(park_names)

# 6. Save the results to a clean Excel file
output_file = "/content/Ocala_Teal_Winter_Data.xlsx"
df_filtered.to_excel(output_file, index=False)

print("\n--------------------------------------------------")
print(f"🏆 SUCCESS! Extracted {len(df_filtered)} Green-winged Teal rows.")
print(f"Your clean file is saved at: {output_file}")
print("--------------------------------------------------")

# Print a quick preview of your data counts by park
if len(df_filtered) > 0:
    print("\nChecklist rows found per park within your date range:")
    print(df_filtered['Park Name'].value_counts())

Reading and filtering eBird data... Please wait.

--------------------------------------------------
🏆 SUCCESS! Extracted 0 Green-winged Teal rows.
Your clean file is saved at: /content/Ocala_Teal_Winter_Data.xlsx
--------------------------------------------------


In [ ]:
import pandas as pd

file_path = "/content/ebd_US-FL-083_gnwtea_202410_202604_unv_smp_relMay-2026_unvetted.txt"

df = pd.read_csv(file_path, sep='\t', usecols=['LOCALITY ID', 'OBSERVATION DATE', 'PROTOCOL CODE'])

print("Total rows in your raw eBird file:", len(df))

# Check unique locations in the file
unique_locs = df['LOCALITY ID'].astype(str).str.strip().unique()
print("\nFirst 10 unique Locality IDs found in your file:")
print(unique_locs[:10])

# Check if your specific target parks are even in this file
target_parks = ['L9783145', 'L35493207']
print("\nAre your target parks in this file?")
for park in target_parks:
    exists = park in unique_locs
    print(f"Park {park}: {'✅ YES' if exists else '❌ NO'}")

# Check the range of dates actually present
print("\nEarliest date in file:", df['OBSERVATION DATE'].min())
print("Latest date in file:", df['OBSERVATION DATE'].max())

Total rows in your raw eBird file: 0

First 10 unique Locality IDs found in your file:
[]

Are your target parks in this file?
Park L9783145: ❌ NO
Park L35493207: ❌ NO

Earliest date in file: nan
Latest date in file: nan


In [ ]:
import os

file_path = ""
file_size = os.path.getsize(file_path)

print(f"File size: {file_size} bytes ({file_size / 1024:.2f} KB)")

In [ ]:
import pandas as pd
import numpy as np

file_path = "/content/Ocala_Teal_Winter_Data.xlsx"
df_clean = pd.read_excel(file_path)

# Ensure date column is treated properly and extract the month/year
df_clean['OBSERVATION DATE'] = pd.to_datetime(df_clean['OBSERVATION DATE'])
df_clean['Month'] = df_clean['OBSERVATION DATE'].dt.strftime('%Y-%m (%B)')

# Clean up the observation counts (convert 'X' presence markers to 1, though numeric counts are expected)
df_clean['OBSERVATION COUNT'] = pd.to_numeric(df_clean['OBSERVATION COUNT'].astype(str).str.replace('X', '1'), errors='coerce').fillna(1)

# Calculate Party Hours for each individual checklist
df_clean['Party Hours'] = df_clean['DURATION MINUTES'] / 60.0

# Group by Park and Month to get totals
monthly_summary = df_clean.groupby(['Park Name', 'Month']).agg(
    Total_Birds=('OBSERVATION COUNT', 'sum'),
    Total_Party_Hours=('Party Hours', 'sum'),
    Checklist_Count=('GLOBAL UNIQUE IDENTIFIER', 'count')
).reset_index()

# Calculate the final Birds Per Party Hour
monthly_summary['Birds Per Party Hour'] = (monthly_summary['Total_Birds'] / monthly_summary['Total_Party_Hours']).round(2)

print("📊 FINAL RESULTS FOR YOUR FIELD PAPER:")
print("-------------------------------------------------------------------------")
print(monthly_summary.to_string(index=False))
print("-------------------------------------------------------------------------")
print("\n📝 NOTE FOR TUSCAWILLA PARK:")
print("Tuscawilla Park had 0.00 Birds Per Party Hour across all months (Nov-Mar)")
print("because Green-winged Teals were completely absent from all eligible checklists.")

📊 FINAL RESULTS FOR YOUR FIELD PAPER:
-------------------------------------------------------------------------
                  Park Name             Month  Total_Birds  Total_Party_Hours  Checklist_Count  Birds Per Party Hour
Ocala Wetland Recharge Park 2025-01 (January)            7          10.166667                6                  0.69
-------------------------------------------------------------------------

📝 NOTE FOR TUSCAWILLA PARK:
Tuscawilla Park had 0.00 Birds Per Party Hour across all months (Nov-Mar)
because Green-winged Teals were completely absent from all eligible checklists.


In [26]:
import pandas as pd
import numpy as np

# 1. File Path Setup
file_path = "/content/ebd_US-FL-083_gnwtea_202410_202604_unv_smp_relMay-2026.txt"

print("Step 1: Reading raw eBird file...")
df = pd.read_csv(file_path, sep='\t', parse_dates=['OBSERVATION DATE'])

# 2. Location Filtering
target_parks = ['L9783145', 'L35493207']
df_filtered = df[df['LOCALITY ID'].astype(str).str.strip().isin(target_parks)].copy()

# 3. Date Filtering (Nov 1, 2024 - Mar 31, 2025)
start_date = '2024-11-01'
end_date = '2025-03-31'
df_filtered = df_filtered[(df_filtered['OBSERVATION DATE'] >= start_date) & (df_filtered['OBSERVATION DATE'] <= end_date)]

# 4. Protocol Filtering (Stationary 'P21' and Traveling 'P22')
df_filtered = df_filtered[df_filtered['PROTOCOL CODE'].isin(['P21', 'P22'])]

# 5. Map real names to Park IDs
park_names = {'L9783145': 'Ocala Wetland Recharge Park', 'L35493207': 'Tuscawilla Park'}
df_filtered['Park Name'] = df_filtered['LOCALITY ID'].map(park_names)

# 6. Save Extracted Rows to Excel
output_file = "/content/Ocala_Teal_Winter_Data.xlsx"
df_filtered.to_excel(output_file, index=False)
print(f"🏆 SUCCESS: Extracted {len(df_filtered)} rows to {output_file}")

# 7. Calculate Birds Per Party Hour (BPPH)
if len(df_filtered) > 0:
    df_filtered['Month'] = df_filtered['OBSERVATION DATE'].dt.strftime('%Y-%m (%B)')
    df_filtered['OBSERVATION COUNT'] = pd.to_numeric(df_filtered['OBSERVATION COUNT'].astype(str).str.replace('X', '1'), errors='coerce').fillna(1)
    df_filtered['Party Hours'] = df_filtered['DURATION MINUTES'] / 60.0

    monthly_summary = df_filtered.groupby(['Park Name', 'Month']).agg(
        Total_Birds=('OBSERVATION COUNT', 'sum'),
        Total_Party_Hours=('Party Hours', 'sum'),
        Checklist_Count=('GLOBAL UNIQUE IDENTIFIER', 'count')
    ).reset_index()

    monthly_summary['Birds Per Party Hour'] = (monthly_summary['Total_Birds'] / monthly_summary['Total_Party_Hours']).round(2)
    print("\n📊 FINAL RESULTS FOR FIELD PAPER:")
    print(monthly_summary.to_string(index=False))

Step 1: Reading raw eBird file...
🏆 SUCCESS: Extracted 6 rows to /content/Ocala_Teal_Winter_Data.xlsx

📊 FINAL RESULTS FOR FIELD PAPER:
                  Park Name             Month  Total_Birds  Total_Party_Hours  Checklist_Count  Birds Per Party Hour
Ocala Wetland Recharge Park 2025-01 (January)            7          10.166667                6                  0.69
